# Lab 9: Luong Attention (Multiplicative Attention)




Theory

Luong Attention (Multiplicative Attention)

Luong Attention is an attention mechanism used in Sequence-to-Sequence (Seq2Seq) models to improve machine translation and other sequence prediction tasks. Instead of relying only on the final hidden state of the encoder, the decoder dynamically focuses on the most relevant encoder hidden states while generating each output word. This enables the model to better capture long-range dependencies and improves translation accuracy.

The model consists of three main components:

Encoder: Converts the input sentence into a sequence of hidden states using an embedding layer followed by a Recurrent Neural Network (RNN). The embeddings learn meaningful vector representations of words during training.

Luong Dot Attention: Computes attention scores by taking the dot product between the current decoder hidden state (query) and each encoder hidden state (keys). The scores are normalized using the Softmax function to obtain attention weights. These weights are used to calculate a context vector, which contains the most relevant information from the input sequence.

Decoder: Receives the context vector along with the previous output word to generate the next word in the target sequence. The decoder repeats this process until the End-of-Sequence (EOS) token is produced.

Before training, the dataset is preprocessed by converting sentences into word indices, building vocabularies (word2index, index2word, and word2count), and converting each sentence into input and target tensors with an EOS token appended. During training, the model minimizes prediction error using backpropagation, and teacher forcing may be used to improve learning. During evaluation, the trained model predicts the translated sentence one word at a time while dynamically attending to different parts of the input sentence.

The Luong Attention mechanism is computationally efficient because it uses multiplicative (dot-product) scoring, making it faster than additive attention while maintaining high translation performance for sequence-to-sequence tasks.

**Requirements**

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data



- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
PATH = r'data\eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes fort contrariee', 'you re very upset']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

### Encoder

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

In [22]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = LuongDotAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [11]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [12]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [15]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [16]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [24]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 3s (- 2m 24s) (5 2%) 1.8891
0m 6s (- 2m 10s) (10 5%) 1.0868
0m 10s (- 2m 3s) (15 7%) 0.7357
0m 13s (- 1m 58s) (20 10%) 0.5060
0m 16s (- 1m 54s) (25 12%) 0.3545
0m 19s (- 1m 51s) (30 15%) 0.2766
0m 22s (- 1m 48s) (35 17%) 0.2066
0m 26s (- 1m 44s) (40 20%) 0.1836
0m 29s (- 1m 41s) (45 22%) 0.1673
0m 33s (- 1m 39s) (50 25%) 0.1372
0m 36s (- 1m 35s) (55 27%) 0.1279
0m 39s (- 1m 33s) (60 30%) 0.1240
0m 43s (- 1m 30s) (65 32%) 0.1199
0m 46s (- 1m 26s) (70 35%) 0.1095
0m 49s (- 1m 22s) (75 37%) 0.1049
0m 52s (- 1m 19s) (80 40%) 0.0951
0m 56s (- 1m 15s) (85 42%) 0.1008
0m 59s (- 1m 12s) (90 45%) 0.0989
1m 2s (- 1m 9s) (95 47%) 0.0926
1m 6s (- 1m 6s) (100 50%) 0.0864
1m 10s (- 1m 4s) (105 52%) 0.0877
1m 15s (- 1m 1s) (110 55%) 0.0794
1m 19s (- 0m 58s) (115 57%) 0.0803
1m 23s (- 0m 55s) (120 60%) 0.0779
1m 27s (- 0m 52s) (125 62%) 0.0714
1m 31s (- 0m 49s) (130 65%) 0.07

In [25]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> t es plantee
= you re stuck
< you re stuck <EOS>

> je suis americaine
= i am an american
< i m faster <EOS>

> nous sommes normales
= we re normal
< we re normal <EOS>

> nous sommes perdues
= we re lost
< we re lost <EOS>

> tu es tres marrante
= you re very funny
< you re very funny <EOS>

> vous etes fort genereuses
= you re very generous
< you re very generous <EOS>

> je suis humble
= i m humble
< i m humble <EOS>

> vous etes trop maigre
= you re too skinny
< you re too skinny <EOS>

> j ai la frousse
= i m jittery
< i m jittery <EOS>

> je me meurs
= i m dying
< she is getting old <EOS>



Discussion

In this lab, a Seq2Seq Neural Machine Translation model with Luong (Multiplicative) Attention was implemented. The dataset was preprocessed and converted into tensors before training. The attention mechanism allowed the decoder to focus on relevant encoder outputs while generating each word, improving translation quality and handling longer sentences more effectively than a basic Seq2Seq model.

Conclusion

This lab successfully implemented a Seq2Seq model with Luong Attention for neural machine translation. The attention mechanism improved translation by allowing the decoder to focus on relevant parts of the input sequence during prediction.